# 04 — Subplots

Poner varios gráficos en la misma figura. Imprescindible para comparar distribuciones, mostrar múltiples métricas al mismo tiempo, o construir dashboards estáticos.

## Setup

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT  = find_project_root()
TRAIN = ROOT / 'data' / 'external' / 'train.csv'
AIR   = ROOT / 'data' / 'external' / 'Listings.csv'

df = pd.read_csv(TRAIN, low_memory=False)
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')
air = pd.read_csv(AIR, encoding='latin-1', low_memory=False)
air_clean = air[(air['price'] > 0) & (air['price'] < 500)].dropna(
    subset=['price', 'accommodates', 'review_scores_rating']
)


## `plt.subplots(nrows, ncols)` — rejilla de gráficos

In [ ]:
# 1 fila, 2 columnas → ax es un array de 2 elementos
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# axes[0] — primer gráfico
ventas_region = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
axes[0].bar(ventas_region.index, ventas_region.values, color='steelblue', edgecolor='white')
axes[0].set_title('Revenue por región')
axes[0].set_ylabel('USD')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[0].grid(axis='y', linestyle=':', alpha=0.4)
axes[0].set_axisbelow(True)

# axes[1] — segundo gráfico
ventas_cat = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
axes[1].bar(ventas_cat.index, ventas_cat.values, color='darkorange', edgecolor='white')
axes[1].set_title('Revenue por categoría')
axes[1].set_ylabel('USD')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[1].grid(axis='y', linestyle=':', alpha=0.4)
axes[1].set_axisbelow(True)

fig.suptitle('Resumen de ventas — Superstore', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## Rejilla 2×2

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# [0,0] — ventas mensuales
ventas_mes = df.groupby(df['Order Date'].dt.to_period('M').astype(str))['Sales'].sum()
axes[0, 0].plot(ventas_mes.index, ventas_mes.values, color='steelblue', linewidth=1.5)
axes[0, 0].set_title('Ventas mensuales')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].grid(axis='y', linestyle=':', alpha=0.4)

# [0,1] — top 10 sub-categorías
top_sub = df.groupby('Sub-Category')['Sales'].sum().sort_values().tail(10)
axes[0, 1].barh(top_sub.index, top_sub.values, color='seagreen', edgecolor='white')
axes[0, 1].set_title('Top 10 sub-categorías')
axes[0, 1].grid(axis='x', linestyle=':', alpha=0.4)
axes[0, 1].set_axisbelow(True)

# [1,0] — histograma precio Airbnb
axes[1, 0].hist(air_clean['price'], bins=50, color='darkorange', edgecolor='white', linewidth=0.3)
axes[1, 0].set_title('Distribución de precio — Airbnb')
axes[1, 0].set_xlabel('USD/noche')
axes[1, 0].grid(axis='y', linestyle=':', alpha=0.4)

# [1,1] — scatter price vs accommodates
axes[1, 1].scatter(air_clean['accommodates'], air_clean['price'],
                   alpha=0.2, s=10, color='tomato', edgecolors='none')
axes[1, 1].set_title('Precio vs accommodates')
axes[1, 1].set_xlabel('Accommodates')
axes[1, 1].set_ylabel('Precio (USD)')
axes[1, 1].grid(linestyle=':', alpha=0.4)

fig.suptitle('Dashboard de análisis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## `sharey` y `sharex` — ejes compartidos para comparar

In [ ]:
# sharey=True: todos los subplots comparten el mismo eje Y
# Imprescindible para comparar magnitudes entre subplots
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)

categorias = df['Category'].unique()
colores = ['steelblue', 'darkorange', 'seagreen']

for i, (cat, color) in enumerate(zip(categorias, colores)):
    ventas = (
        df[df['Category'] == cat]
        .groupby(df['Order Date'].dt.to_period('M').astype(str))['Sales']
        .sum()
    )
    axes[i].plot(ventas.index, ventas.values, color=color, linewidth=2)
    axes[i].set_title(cat)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].grid(axis='y', linestyle=':', alpha=0.4)
    if i == 0:
        axes[i].set_ylabel('Revenue (USD)')

fig.suptitle('Ventas mensuales por categoría — escala compartida', fontsize=12)
plt.tight_layout()
plt.show()


## `gridspec_kw` — subplots con tamaños distintos

In [ ]:
# width_ratios define el ancho relativo de cada columna
fig, axes = plt.subplots(1, 2, figsize=(12, 4),
                          gridspec_kw={'width_ratios': [2, 1]})

# Gráfico principal — ocupa 2/3 del ancho
ventas_mes = df.groupby(df['Order Date'].dt.to_period('M').astype(str))['Sales'].sum()
axes[0].plot(ventas_mes.index, ventas_mes.values, color='steelblue', linewidth=2)
axes[0].set_title('Ventas mensuales')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', linestyle=':', alpha=0.4)

# Gráfico secundario — ocupa 1/3 del ancho
segmento = df.groupby('Segment')['Sales'].sum().sort_values(ascending=False)
axes[1].bar(segmento.index, segmento.values, color='darkorange', edgecolor='white')
axes[1].set_title('Revenue por segmento')
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(axis='y', linestyle=':', alpha=0.4)
axes[1].set_axisbelow(True)

plt.tight_layout()
plt.show()


---
## Resumen

| Patrón | Sintaxis |
|--------|----------|
| 1 fila, N cols | `fig, axes = plt.subplots(1, N, figsize=...)` |
| N filas, M cols | `fig, axes = plt.subplots(N, M)` → `axes[i, j]` |
| Eje Y compartido | `sharey=True` |
| Eje X compartido | `sharex=True` |
| Anchos distintos | `gridspec_kw={'width_ratios': [2, 1]}` |
| Título de figura | `fig.suptitle(...)` |
